# µhLS Backend Cookbook

This guided tour takes one compact dataflow kernel through the HLS backend: allocation chooses implementation classes and timing, scheduling assigns control steps, and binding creates concrete functional-unit and register resources.

The examples are complete demonstrations rather than laboratory exercises. They expose the decisions and intermediate artifacts so you can change the kernel or cost model and rerun the pipeline.

## Setup

Run this cell first. It locates the repository root, adds src/ to Python's import path, and imports the backend APIs used below.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "uhls").exists():
    repo_root = repo_root.parent

if not (repo_root / "src" / "uhls").exists():
    raise RuntimeError("could not locate repo root containing src/uhls")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from IPython.display import display

from uhls.frontend import lower_source_to_uir
from graphviz import Source
from uhls.middleend.passes.opt import CSEPass, ConstPropPass, CopyPropPass, DCEPass
from uhls.middleend.passes.util import PassContext, PassManager
from uhls.middleend.uir import COMPACT_OPCODE_LABELS, format_module, verify_module
from uhls.backend.hls import (
    ExecutabilityGraph,
    bind_dump_to_dot,
    format_bind_dump,
    lower_alloc_to_sched,
    lower_module_to_seq,
    lower_sched_to_bind,
    lower_seq_to_alloc,
)
from uhls.backend.hls.alloc import executability_graph_to_dot
from uhls.backend.hls.uhir import format_uhir, to_dot


def show_design(design, max_lines=100):
    lines = format_uhir(design).splitlines()
    print("\n".join(lines[:max_lines]))
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} more lines)")


def show_dot(title, dot):
    print(title)
    display(Source(dot))


print(f"repo_root = {repo_root}")

## 1. Build a sequencing graph

The kernel has two independent additions followed by a multiply. That small fork-and-join shape is useful because it exposes parallelism, a multi-cycle operation, and a later resource-sharing decision.

The frontend and µIR cleanup are included only to provide a realistic input. The backend begins at seq-stage µhIR.

In [ ]:
BACKEND_C_SOURCE = """
int32_t resource_demo(int32_t a, int32_t b, int32_t c) {
    int32_t left = a + b;
    int32_t right = b + c;
    return left * right;
}
"""

uir_module = lower_source_to_uir(BACKEND_C_SOURCE)
verify_module(uir_module)
canonical_uir = PassManager(
    [CSEPass(), CopyPropPass(), ConstPropPass(), DCEPass()]
).run(deepcopy(uir_module), PassContext())
verify_module(canonical_uir)

seq_design = lower_module_to_seq(canonical_uir, top="resource_demo")

print("Canonical µIR:\n")
print(format_module(canonical_uir))
print("\nseq-stage µhIR:\n")
show_design(seq_design)
show_dot("Input sequencing graph", to_dot(seq_design, compact=True))

## 2. Allocation: choose implementation classes

An executability graph is a weighted compatibility relation. Each edge says that a functional-unit class can execute an opcode with a particular initiation interval and delay.

Here, additions use a one-cycle DEMO_ALU and multiplication uses a two-cycle DEMO_MUL. EWMS is a generic fallback for all other non-structural operations in this demonstration. Allocation annotates operations with class, ii, and delay; it does not yet create individual hardware instances.

In [ ]:
all_operations = tuple(sorted(COMPACT_OPCODE_LABELS))
specialized_operations = {"add", "mul"}
fallback_edges = tuple(
    ("EWMS", operation, 1, 1)
    for operation in all_operations
    if operation not in specialized_operations
)
operation_edges = (
    ("DEMO_ALU", "add", 1, 1),
    ("DEMO_MUL", "mul", 1, 2),
)

executability_graph = ExecutabilityGraph(
    functional_units=("EWMS", "DEMO_ALU", "DEMO_MUL"),
    operations=all_operations,
    edges=fallback_edges + operation_edges,
)

alloc_design = lower_seq_to_alloc(
    seq_design,
    executability_graph=executability_graph,
    algorithm="min_delay",
)

print("Allocated µhIR:\n")
show_design(alloc_design)

alloc_region = next(region for region in alloc_design.regions if region.kind == "procedure")
print("\nAllocated operation table:")
print(f"{'node':<8} {'opcode':<10} {'class':<12} {'ii':>3} {'delay':>6}")
for node in alloc_region.nodes:
    print(
        f"{node.id:<8} {node.opcode:<10} "
        f"{str(node.attributes.get('class')):<12} "
        f"{str(node.attributes.get('ii')):>3} "
        f"{str(node.attributes.get('delay')):>6}"
    )

show_dot("Executability graph", executability_graph_to_dot(executability_graph))
show_dot("Allocated dataflow graph", to_dot(alloc_design, compact=True))

### Allocation is not binding

DEMO_ALU names a kind of implementation, not a single adder. Both additions may be classified as DEMO_ALU even if their overlapping schedules later require two concrete adder instances. Binding makes that instance-level choice.

## 3. Scheduling: assign control steps

The built-in ASAP scheduler starts each operation as soon as all data predecessors are ready. µhLS uses zero-based control steps. A delay-2 multiply starting at step 1 occupies the inclusive interval [1, 2], and its result becomes available afterward.

ASAP considers dependencies and operation delays. A resource-constrained list scheduler can deliberately delay a ready operation when the available instance count is smaller than the parallel demand.

In [ ]:
sched_design = lower_alloc_to_sched(alloc_design, algorithm="asap")
sched_region = next(region for region in sched_design.regions if region.kind == "procedure")

print("Scheduled µhIR:\n")
show_design(sched_design)

print("\nSchedule table:")
print(f"{'node':<8} {'opcode':<10} {'class':<12} {'start':>5} {'end':>5}")
for node in sched_region.nodes:
    print(
        f"{node.id:<8} {node.opcode:<10} "
        f"{str(node.attributes.get('class')):<12} "
        f"{str(node.attributes.get('start')):>5} "
        f"{str(node.attributes.get('end')):>5}"
    )

add_nodes = [node for node in sched_region.nodes if node.opcode == "add"]
mul_nodes = [node for node in sched_region.nodes if node.opcode == "mul"]
assert len(add_nodes) == 2 and len(mul_nodes) == 1
assert len({node.attributes["start"] for node in add_nodes}) == 1
assert mul_nodes[0].attributes["end"] - mul_nodes[0].attributes["start"] + 1 == 2

print("\nregion latency:", sched_region.latency)
show_dot("ASAP-scheduled dataflow graph", to_dot(sched_design, compact=True))

## 4. Binding: create concrete resources

Left-edge binding treats each scheduled operation as an interval. Overlapping intervals of the same class conflict and need different functional-unit instances; non-overlapping intervals may share one. Value live ranges are colored similarly to allocate registers.

Because the two additions start together, this schedule needs two DEMO_ALU instances. A later non-overlapping DEMO_ALU operation could reuse either instance.

In [ ]:
bind_design = lower_sched_to_bind(sched_design, algorithm="left_edge")
bind_region = next(region for region in bind_design.regions if region.kind == "procedure")

print("Bound µhIR:\n")
show_design(bind_design)

fu_resources = [resource for resource in bind_design.resources if resource.kind == "fu"]
register_resources = [resource for resource in bind_design.resources if resource.kind == "reg"]

print("\nFunctional-unit resources:")
for resource in fu_resources:
    print(f"  {resource.id}: {resource.value}")

print("\nRegister resources:")
for resource in register_resources:
    print(f"  {resource.id}: {resource.value}")

print("\nOperation bindings:")
for node in bind_region.nodes:
    binding = node.attributes.get("bind")
    if binding is not None:
        print(f"  {node.id} ({node.opcode}) -> {binding}")

print("\nValue live ranges:")
for binding in bind_region.value_bindings:
    print(f"  {binding.producer} -> {binding.register} {binding.live_intervals}")

alu_resources = [resource for resource in fu_resources if resource.value == "DEMO_ALU"]
assert len(alu_resources) == 2

## 5. Inspect resource maps

The DFGSB combines dataflow, schedule, and binding. The time-resource plane reorganizes the same information by control step and concrete hardware instance. Conflict graphs explain why operations or values cannot share a resource.

These views are generated from the bound µhIR; they are diagnostics, not separate source-of-truth files.

In [ ]:
print("Time-resource plane (text):\n")
print(format_bind_dump(bind_design, ("trp",), compact=True))

show_dot(
    "Dataflow graph with schedule and binding",
    bind_dump_to_dot(bind_design, ("dfgsb",), compact=True),
)
show_dot(
    "Time-resource plane",
    bind_dump_to_dot(bind_design, ("trp",), compact=True),
)
show_dot(
    "Binding conflict graph",
    bind_dump_to_dot(bind_design, ("conflict",), compact=True),
)

## Takeaways

- Allocation selects compatible implementation classes and attaches timing costs.
- Scheduling maps dependency-constrained operations onto control-step intervals.
- Resource limits can trade parallelism for latency.
- Binding turns abstract classes into concrete functional-unit instances and registers.
- DFGSB, conflict, and time-resource views make those decisions auditable before RTL lowering.